In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
from datetime import datetime

RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

# load schedule
schedule = pd.read_csv(RAW_DIR / "wc2026_schedule.csv")

results_path = RAW_DIR / "wc2026_results.csv"

# generate template if file doesn't exist
if not results_path.exists():
    print("No results file found — generating template...")
    rows = []
    for _, row in schedule.iterrows():
        if not row["home_team"] or not row["away_team"]:
            continue
        if str(row["home_team"]).startswith("Win") or \
           str(row["home_team"]).startswith("2nd"):
            continue
        rows.append({
            "match_id":   row["match_id"],
            "date":       row["date"],
            "group":      row["group"],
            "stage":      row["stage"],
            "matchday":   row["matchday"],
            "home_team":  row["home_team"],
            "home_score": "",
            "away_team":  row["away_team"],
            "away_score": "",
        })
    template_df = pd.DataFrame(rows)
    template_df.to_csv(results_path, index=False)
    print(f"Template created → {results_path}")
    print(f"Total matches: {len(template_df)}")
    print()
    print("Open data/raw/wc2026_results.csv and fill home_score and away_score")
    print("after each match. Leave future matches blank.")
else:
    print(f"Results file found → {results_path}")

# load results
results = pd.read_csv(results_path)

# show current state
completed = results[
    results["home_score"].notna() & 
    (results["home_score"].astype(str).str.strip() != "")
]
pending   = results[results["home_score"].isna() | 
                    results["home_score"].astype(str).str.strip() == ""]

print(f"\nCompleted: {len(completed)}")
print(f"Pending:   {len(pending)}")
print()
if len(completed) > 0:
    print("Last entered result:")
    last = completed.iloc[-1]
    print(f"  {last['home_team']} {int(last['home_score'])}-"
          f"{int(last['away_score'])} {last['away_team']}")

Results file found → ..\data\raw\wc2026_results.csv

Completed: 48
Pending:   0

Last entered result:
  Colombia 1-0 DR Congo


In [2]:
# load current elo
elo_ratings = pd.read_csv(PROCESSED_DIR / "elo_ratings_current.csv")
elo_dict = dict(zip(elo_ratings["team"], elo_ratings["elo"]))

def update_elo(home, away, home_score, away_score, elo_dict, k=60):
    home_elo = elo_dict.get(home, 1500)
    away_elo = elo_dict.get(away, 1500)

    home_exp = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    away_exp = 1 - home_exp

    if home_score > away_score:
        home_act, away_act = 1.0, 0.0
    elif home_score < away_score:
        home_act, away_act = 0.0, 1.0
    else:
        home_act, away_act = 0.5, 0.5

    elo_dict[home] = home_elo + k * (home_act - home_exp)
    elo_dict[away] = away_elo + k * (away_act - away_exp)
    return elo_dict

print("Elo function ready ✓")

Elo function ready ✓


In [3]:
# find rows that have scores filled but outcome not yet computed
# outcome column may not exist yet — add it if missing
if "outcome" not in results.columns:
    results["outcome"] = ""

# find completed rows without outcome yet processed
to_process = results[
    (results["home_score"].notna()) &
    (results["home_score"].astype(str).str.strip() != "") &
    (results["outcome"].isna() | (results["outcome"].astype(str).str.strip() == ""))
].copy()

print(f"New results to process: {len(to_process)}")
print()

if len(to_process) == 0:
    print("Nothing new to process.")
    print("Fill in scores in wc2026_results.csv and re-run.")
else:
    for idx, row in to_process.iterrows():
        home_score = int(row["home_score"])
        away_score = int(row["away_score"])

        # outcome
        if home_score > away_score:   outcome = "home_win"
        elif home_score < away_score: outcome = "away_win"
        else:                          outcome = "draw"

        results.at[idx, "outcome"] = outcome

        # winner for display
        if outcome == "home_win":   winner = row["home_team"]
        elif outcome == "away_win": winner = row["away_team"]
        else:                        winner = "Draw"

        # update elo
        old_home = round(elo_dict.get(row["home_team"], 1500), 1)
        old_away = round(elo_dict.get(row["away_team"], 1500), 1)
        elo_dict = update_elo(
            row["home_team"], row["away_team"],
            home_score, away_score, elo_dict
        )
        new_home = round(elo_dict[row["home_team"]], 1)
        new_away = round(elo_dict[row["away_team"]], 1)

        print(f"  {row['date']} | {row['home_team']} "
              f"{home_score}-{away_score} {row['away_team']}"
              f" → {winner}")
        print(f"    Elo: {row['home_team']}: {old_home} → {new_home} "
              f"({new_home-old_home:+.1f})")
        print(f"    Elo: {row['away_team']}: {old_away} → {new_away} "
              f"({new_away-old_away:+.1f})")
        print()

    # save results with outcomes
    results.to_csv(results_path, index=False)
    print(f"Results saved → {results_path}")

    # save updated elo
    elo_df = pd.DataFrame([
        {"team": t, "elo": round(r, 1)}
        for t, r in elo_dict.items()
    ]).sort_values("elo", ascending=False).reset_index(drop=True)
    elo_df.index += 1
    elo_df.to_csv(PROCESSED_DIR / "elo_ratings_current.csv", index=False)
    print(f"Elo updated → elo_ratings_current.csv")

New results to process: 0

Nothing new to process.
Fill in scores in wc2026_results.csv and re-run.


In [4]:
# find which matchdays have all results filled
completed_all = results[
    results["home_score"].notna() &
    (results["home_score"].astype(str).str.strip() != "")
]

matchday_totals = results.groupby("matchday").size()
matchday_done   = completed_all.groupby("matchday").size()

print("Matchday completion status:")
print("-" * 45)
for md in matchday_totals.index:
    total = matchday_totals[md]
    done  = matchday_done.get(md, 0)
    pct   = done / total * 100
    bar   = "█" * done + "░" * (total - done)
    print(f"  {md:<20} {done:>2}/{total}  {bar}  {pct:.0f}%")

print()

# load prediction files and compare
PREDS_DIR = Path("../data/predictions")
correct_total = 0
predicted_total = 0

for md in matchday_done.index:
    filename = f"predictions_{md.lower().replace(' ', '_')}.json"
    pred_path = PREDS_DIR / filename

    if not pred_path.exists():
        continue

    with open(pred_path) as f:
        preds_data = json.load(f)

    # build results lookup
    md_results = completed_all[completed_all["matchday"] == md]
    results_lookup = {
        str(r["match_id"]): r
        for _, r in md_results.iterrows()
    }

    correct = 0
    total   = 0

    print(f"--- {md} predictions vs actual ---")
    for p in preds_data["predictions"]:
        mid = str(p["match_id"])
        if mid not in results_lookup:
            continue

        actual   = results_lookup[mid]
        h_score  = int(actual["home_score"])
        a_score  = int(actual["away_score"])
        act_out  = actual["outcome"]
        pred_out = p["predicted"]

        match = "✓" if pred_out == act_out else "✗"
        if pred_out == act_out: correct += 1
        total += 1

        if act_out == "home_win":   act_winner = actual["home_team"]
        elif act_out == "away_win": act_winner = actual["away_team"]
        else:                        act_winner = "Draw"

        if pred_out == "home_win":   pred_winner = p["home_team"]
        elif pred_out == "away_win": pred_winner = p["away_team"]
        else:                         pred_winner = "Draw"

        print(f"  {match} {p['home_team']} vs {p['away_team']}")
        print(f"     Result:    {h_score}-{a_score} ({act_winner})")
        print(f"     Predicted: {pred_winner} "
              f"({float(p['home_win_prob'])*100:.0f}% / "
              f"{float(p['draw_prob'])*100:.0f}% / "
              f"{float(p['away_win_prob'])*100:.0f}%)")

    print(f"  Accuracy: {correct}/{total} = "
          f"{correct/total*100:.1f}%" if total > 0 else "  No results yet")
    print()
    correct_total   += correct
    predicted_total += total

if predicted_total > 0:
    print(f"Overall accuracy: {correct_total}/{predicted_total} = "
          f"{correct_total/predicted_total*100:.1f}%")

Matchday completion status:
---------------------------------------------
  Final                 0/1  ░  0%
  Matchday 1           24/24  ████████████████████████  100%
  Matchday 2           24/24  ████████████████████████  100%
  Matchday 3            0/24  ░░░░░░░░░░░░░░░░░░░░░░░░  0%
  Quarterfinal          0/4  ░░░░  0%
  Round of 16           0/8  ░░░░░░░░  0%
  Round of 32           0/16  ░░░░░░░░░░░░░░░░  0%
  Semifinal             0/2  ░░  0%
  Third Place           0/1  ░  0%

--- Matchday 1 predictions vs actual ---
  ✓ Mexico vs South Africa
     Result:    2-0 (Mexico)
     Predicted: Mexico (69% / 22% / 9%)
  ✓ South Korea vs Czech Republic
     Result:    2-1 (South Korea)
     Predicted: South Korea (58% / 25% / 16%)
  ✗ Canada vs Bosnia and Herzegovina
     Result:    1-1 (Draw)
     Predicted: Canada (77% / 16% / 6%)
  ✓ United States vs Paraguay
     Result:    4-1 (United States)
     Predicted: United States (49% / 24% / 27%)
  ✗ Qatar vs Switzerland
     Result: 

In [5]:
print("Retraining check:")
print("-" * 45)

retrain_rules = {
    "Matchday 1": 24,
    "Matchday 2": 24,
    "Matchday 3": 24,
}
for md, required in retrain_rules.items():
    done  = matchday_done.get(md, 0)
    total = matchday_totals.get(md, required)
    if done >= required:
        print(f"  ✓ {md} complete ({done}/{required})"
              f" → RETRAIN before next matchday")
        print(f"    Run 05_train_model.ipynb now")
    else:
        print(f"  ○ {md}: {done}/{required} results entered")

print()

# update accuracy.json for frontend
accuracy_path = Path("../docs/accuracy.json")
accuracy = {
    "total_predicted": predicted_total,
    "total_correct":   correct_total,
    "accuracy":        round(correct_total / predicted_total, 3)
                       if predicted_total > 0 else 0,
    "by_matchday":     [],
    "last_updated":    datetime.now().isoformat()
}

for md in matchday_done.index:
    filename  = f"predictions_{md.lower().replace(' ', '_')}.json"
    pred_path = PREDS_DIR / filename
    if not pred_path.exists():
        continue
    with open(pred_path) as f:
        preds_data = json.load(f)

    md_results = completed_all[completed_all["matchday"] == md]
    results_lookup = {str(r["match_id"]): r for _, r in md_results.iterrows()}

    correct = sum(
        1 for p in preds_data["predictions"]
        if str(p["match_id"]) in results_lookup and
        p["predicted"] == results_lookup[str(p["match_id"])]["outcome"]
    )
    total = sum(
        1 for p in preds_data["predictions"]
        if str(p["match_id"]) in results_lookup
    )
    accuracy["by_matchday"].append({
        "matchday": md,
        "correct":  correct,
        "total":    total,
        "accuracy": round(correct/total, 3) if total > 0 else 0
    })

with open(accuracy_path, "w") as f:
    json.dump(accuracy, f, indent=2)

print(f"Accuracy tracker updated → docs/accuracy.json")

Retraining check:
---------------------------------------------
  ✓ Matchday 1 complete (24/24) → RETRAIN before next matchday
    Run 05_train_model.ipynb now
  ✓ Matchday 2 complete (24/24) → RETRAIN before next matchday
    Run 05_train_model.ipynb now
  ○ Matchday 3: 0/24 results entered

Accuracy tracker updated → docs/accuracy.json
